# Processes Non/poisonous Snakebite CSVs into one file

Outputs:
    all_cases_districts.geojson

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
import calendar
import os

In [2]:
proc_data_folder = "proc_data"

p_case_df = gpd.read_file(os.path.join(proc_data_folder, "p_merged_districts.geojson"))
np_case_df = gpd.read_file(os.path.join(proc_data_folder, "np_merged_districts.geojson"))
pop_df = pd.read_csv(os.path.join(proc_data_folder, "pop_dist.csv"), dtype={"district_code": str})

In [ ]:
test_dates = ["2021-02-24", "2021-09-28", "2019-07-28", "2025-07-28"]

for date in test_dates:
    print(p_case_df[p_case_df["DISTRICT"] == "RUPANDEHI"][date] + np_case_df[np_case_df["DISTRICT"] == "RUPANDEHI"][date])
    print(p_case_df[p_case_df["DISTRICT"] == "TAPLEJUNG"][date] + np_case_df[np_case_df["DISTRICT"] == "TAPLEJUNG"][date])

In [4]:
info_cols = [col for col in p_case_df.columns if "20" not in col]
joined_case_df = p_case_df.merge(np_case_df, on=info_cols, how='inner')

In [5]:
case_cols_to_sum = joined_case_df.filter(regex='^20').columns
col_mapping = {col: col.split('_')[0] for col in case_cols_to_sum}
cases_only_df = joined_case_df[case_cols_to_sum].rename(columns=col_mapping)
cases_only_df = cases_only_df.T.reset_index().groupby("index").sum().T.reset_index(drop=True)

In [6]:
case_df = pd.concat([joined_case_df[info_cols], cases_only_df], axis=1)

In [ ]:
# Check for consistency with earlier block
for date in test_dates:
    print(case_df[case_df["DISTRICT"] == "RUPANDEHI"][date])
    print(case_df[case_df["DISTRICT"] == "TAPLEJUNG"][date])

In [19]:
proc_data_folder = "proc_data"
out_file = "all_cases_districts.geojson"

case_df.to_file(os.path.join(proc_data_folder, out_file), driver="GeoJSON")

print(f"\nSaved:")
print(f"  {os.path.join(proc_data_folder, out_file)}")
print("Done.")


Saved:
  proc_data/all_cases_districts.geojson
Done.
